# Data Wrangling

Steps to follow:

1. Remove duplicates
2. Standardize Country
3. Validate Email
4. Remove impossible salaries
5. Flag future hire dates
6. Fill NULLs

## Dirt table
| Problem                 | Rows             |
| ----------------------- | ---------------- |
| Duplicate EmployeeID    | 1 & 5            |
| Missing Email           | 4                |
| Duplicate Email         | 2 & 7            |
| NULL Salary             | 6                |
| Negative Salary         | 8                |
| Salary Outlier          | 9                |
| Future Date             | 10               |
| Invalid Email           | 10               |
| NULL Department         | 10               |
| NULL Role               | 10               |
| Country Standardization | Brazil/BR/Brasil |


In [0]:
%sql
USE CATALOG global_enterprises;
USE SCHEMA bronze;

In [0]:
%sql
SELECT current_schema();

In [0]:
df = spark.read.table('global_enterprises.bronze.employees')

display(df)

In [0]:
import statistics as sta

df.printSchema()

In [0]:
%sql
ALTER TABLE employees ALTER COLUMN EmployeeId SET NOT NULL;

In [0]:
from pyspark.sql.functions import col, when

# Deduplicating
df = df.dropDuplicates()
df = df.dropDuplicates(["EmployeeID"])

display(df)

In [0]:
%sql
with distinct_employees AS (select distinct * from employees)

select count(*) from distinct_employees
union all
select count(*) from employees;

In [0]:
# Standardize countries
from pyspark.sql.functions import col, when, upper

df = (
    df
    .withColumn('Country', when(col('Country') == 'BR', 'Brazil').otherwise(col('Country')))
    .withColumn('Country', upper(col('Country')))
    )

display(df)

In [0]:
df = df.dropna()

display(df)

In [0]:
df = df.withColumn(
    "Salary", when(col("Salary") > 0, col("Salary")).otherwise(col('Salary') * -1)
)

display(df)

In [0]:
# Fixing excessive salaries
import numpy as np
from pyspark.sql.functions import when, col, lit

df = df.withColumn("Salary", when(col("Salary") == 999999.99, lit(np.nan)).otherwise(col("Salary")))

display(df)